# Pipeline VLM Mamografía — Fine-Tuning MedGemma-4b-it
**Autor:** Luis Enrique Medrano Santana  
**Asesor:** Luis Vives Garnique  
**Institución:** Pontificia Universidad Católica del Perú (PUCP)  
**Título:** Modelo generativo multimodal de visión-lenguaje orientado al apoyo a la toma de decisiones clínicas en el diagnóstico de cáncer de mama

---
**Objetivo:** Fine-tuning multi-tarea de MedGemma 4B-IT sobre VinDr-Mammo para predecir simultáneamente:
- `density`: densidad mamaria ACR (A/B/C/D)
- `findings`: descripción textual de hallazgos por vista y lateralidad
- `birads`: categoría BI-RADS (1-5)


## 1. Instalación de dependencias

Versiones fijas para reproducibilidad. `bitsandbytes` habilita cuantización 4-bit NF4 para cargar MedGemma en ~5GB VRAM. `peft` provee LoRA. `trl` provee SFTTrainer.

In [1]:
!pip install -q transformers==4.56.2 peft==0.18.1 trl==0.23.1 accelerate==1.10.1 \
    bitsandbytes==0.48.1 sentencepiece protobuf scikit-learn
print('Deps OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.4 MB/s eta 0:00:00
Deps OK


## 2. Montar Drive y definir rutas

- `PROCESSED_DIR`: PNGs preprocesados desde DICOM (Otsu ROI crop + CLAHE + resize 448x448, merge 896x896)
- `AUG_DIR`: PNGs augmentados por clase (flip horizontal + rotación ±15°) solo para BIRADS 3, 4, 5
- `CHECKPOINT_DIR`: donde se guardan los checkpoints por época
- `LOCAL_IMG_DIR` / `LOCAL_AUG_DIR`: disco local de Colab (mucho más rápido que Drive para I/O de training)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_BASE     = Path('/content/drive/MyDrive/vindr_1000')
METADATA_DIR   = DRIVE_BASE / 'metadata'

# Imagenes preprocesadas (DICOM -> Otsu -> CLAHE -> 448x448 -> merge 896x896)
PROCESSED_DIR  = DRIVE_BASE / 'processed_dcm'

# Augmentadas: subcarpetas BIRADS_3/, BIRADS_4/, BIRADS_5/
# Cada subcarpeta tiene: originales + _flip + _rot (flip prioritario, rot solo si falta)
AUG_DIR        = DRIVE_BASE / 'processed_aug_2'

# Checkpoints del modelo durante training
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints' / 'multitask_vf'

# Disco local Colab: mucho mas rapido que Drive para leer imagenes en cada step
LOCAL_IMG_DIR  = Path('/content/processed')
LOCAL_AUG_DIR  = Path('/content/processed_aug')

(CHECKPOINT_DIR / 'run').mkdir(parents=True, exist_ok=True)
print('Rutas OK')
print(f'  PROCESSED_DIR : {PROCESSED_DIR}')
print(f'  AUG_DIR       : {AUG_DIR}')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')

Mounted at /content/drive
Rutas OK
  PROCESSED_DIR : /content/drive/MyDrive/vindr_1000/processed_dcm
  AUG_DIR       : /content/drive/MyDrive/vindr_1000/processed_aug_2
  CHECKPOINT_DIR: /content/drive/MyDrive/vindr_1000/checkpoints/multitask_vf


## 3. Copiar imágenes a disco local + cache en RAM

**Por qué copiar a disco local:** Drive tiene ~50MB/s de lectura secuencial pero alta latencia por archivo. Durante training, el DataLoader lee ~1 imagen por step — con Drive esto introduce delays de 100-500ms por imagen que escalan a horas. Copiando a disco local primero, la lectura es instantánea.

**Cache en RAM:** Se pre-cargan todas las imágenes en `IMAGE_CACHE` (dict path→PIL). Elimina completamente el I/O durante training. Con 896x896 RGB, ~2200 imágenes ocupan ~5GB RAM — Colab Pro+ tiene 50GB+ disponibles.

In [3]:
import subprocess
from PIL import Image as PILImage
from tqdm import tqdm
import pandas as pd

# --- Copiar processed_dcm a disco local ---
LOCAL_IMG_DIR.mkdir(exist_ok=True)
print('Copiando processed_dcm -> disco local...')
subprocess.run(['rsync', '-a', '--info=progress2',
    str(PROCESSED_DIR)+'/', str(LOCAL_IMG_DIR)+'/'], capture_output=False)

# --- Copiar augmentadas (estructura plana) a disco local ---
# Las augmentadas estan en subcarpetas BIRADS_3/, BIRADS_4/, BIRADS_5/
# Las copiamos a una carpeta plana para simplificar el indexado
LOCAL_AUG_DIR.mkdir(exist_ok=True)
print('Copiando augmentadas -> disco local (aplanando subcarpetas)...')
for b in [3, 4, 5]:
    subdir = AUG_DIR / f'BIRADS_{b}'
    if subdir.exists():
        subprocess.run(['rsync', '-a', str(subdir)+'/', str(LOCAL_AUG_DIR)+'/'],
                       capture_output=False)
        print(f'  BIRADS_{b}: copiado')

def resolve_path(p):
    """Busca primero en disco local, luego en aug local, finalmente usa ruta original."""
    name = Path(p).name
    local = LOCAL_IMG_DIR / name
    if local.exists(): return str(local)
    aug = LOCAL_AUG_DIR / name
    if aug.exists(): return str(aug)
    return str(p)

# --- Cache en RAM: cargar todas las imagenes procesadas ---
IMAGE_CACHE = {}
all_pngs = list(LOCAL_IMG_DIR.glob('*.png')) + list(LOCAL_AUG_DIR.glob('*.png'))
for p in tqdm(all_pngs, desc='Disco->RAM'):
    try:
        IMAGE_CACHE[str(p)] = PILImage.open(p).convert('RGB')
    except:
        pass

print(f'Cache RAM: {len(IMAGE_CACHE)} imagenes')

Copiando processed_dcm -> disco local...
Copiando augmentadas -> disco local (aplanando subcarpetas)...
  BIRADS_3: copiado
  BIRADS_4: copiado
  BIRADS_5: copiado


Disco->RAM: 100%|██████████| 5253/5253 [00:42<00:00, 123.36it/s]

Cache RAM: 5253 imagenes


## 4. Construir df_all — dataset base

Lee los dos CSVs de VinDr-Mammo:
- `breast-level_annotations.csv`: una fila por imagen (4 por estudio), con BIRADS y density por vista
- `finding_annotations.csv`: una fila por finding por imagen, con categoría del hallazgo

Para cada `study_id` que tenga PNG procesado, construye un registro con:
- `breast_birads`: BIRADS máximo entre las 4 vistas (el más severo)
- `density`: primer density_char encontrado (consistente entre vistas)
- `findings`: texto natural concatenando hallazgos por lateralidad+vista
- `report`: JSON string que será el target del modelo

**Nota:** Los estudios augmentados NO se agregan aquí — se incorporan en la celda de rebalanceo.

In [4]:
import ast, json
import pandas as pd

breast_df  = pd.read_csv(METADATA_DIR / 'breast-level_annotations.csv')
finding_df = pd.read_csv(METADATA_DIR / 'finding_annotations.csv')

# Mapeos para texto natural en findings
LATERALITY_FULL = {'R': 'right', 'L': 'left'}
VIEW_FULL = {'CC': 'cranio-caudal (CC)', 'MLO': 'medio-lateral oblique (MLO)'}

def safe_last_alnum(s):
    """Extrae el ultimo caracter alfanumerico de un string (ej. 'BI-RADS 3' -> '3', 'DENSITY C' -> 'C')."""
    if s is None or (isinstance(s, float) and pd.isna(s)): return ''
    for ch in reversed(str(s).strip()):
        if ch.isalnum(): return ch
    return ''

def normalize_lat(x):
    """Normaliza lateralidad a 'L' o 'R'."""
    t = str(x).strip().upper() if x else ''
    return 'L' if t.startswith('L') else ('R' if t.startswith('R') else '')

def normalize_view(x):
    """Normaliza vista a 'CC' o 'MLO'."""
    t = str(x).strip().upper() if x else ''
    return 'MLO' if 'MLO' in t else ('CC' if 'CC' in t else '')

def parse_categories(raw):
    """Parsea finding_categories desde string a lista de strings."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return []
    try:
        out = ast.literal_eval(str(raw).strip())
        return [str(x).strip() for x in out] if isinstance(out, list) else [str(out).strip()]
    except:
        return [x.strip() for x in str(raw).strip('[]').replace("'",'').split(',') if x.strip()]

def build_findings(finding_df, study_id):
    """
    Construye el texto de findings y los flags binarios (mass/calcification/asymmetry)
    a partir de finding_annotations.csv para un study_id dado.
    Formato: '<finding> found in <laterality> <view>' separados por ' and '.
    Si no hay hallazgos: 'Healthy Breast. No Findings'.
    """
    frows = finding_df[finding_df['study_id'] == study_id]
    mass = calc = asym = 0
    if frows.empty:
        return 'Healthy Breast. No Findings', mass, calc, asym
    grouped, all_no_find = {}, True
    for _, row in frows.iterrows():
        lat, view = normalize_lat(row.get('laterality','')), normalize_view(row.get('view_position',''))
        for cat in parse_categories(row.get('finding_categories','[]')):
            cl = cat.lower().strip()
            if not cl: continue
            if 'mass' in cl: mass = 1
            if 'calcification' in cl: calc = 1
            if 'asymmetry' in cl: asym = 1
            if cl != 'no finding': all_no_find = False
            grouped.setdefault((lat,view), {}).setdefault(cat, 0)
            grouped[(lat,view)][cat] += 1
    if all_no_find:
        return 'Healthy Breast. No Findings', mass, calc, asym
    parts = []
    for (lat, view), cats in grouped.items():
        filtered = {c:n for c,n in cats.items() if c.lower() != 'no finding'}
        if not filtered: continue
        lf = LATERALITY_FULL.get(lat, lat.lower())
        vf = VIEW_FULL.get(view, view)
        phrase = ' and '.join(c.lower() for c in filtered)
        parts.append(f'{phrase} found in {lf} {vf}')
    return (' and '.join(parts) if parts else 'Healthy Breast. No Findings'), mass, calc, asym

# Indexar PNGs procesados por study_id
png_index = {p.stem: str(p) for p in LOCAL_IMG_DIR.glob('*.png')}
print(f'PNGs procesados encontrados: {len(png_index)}')

# Construir df_all: un registro por study_id que tenga PNG procesado
records = []
for study_id, sdf in tqdm(breast_df.groupby('study_id'), desc='Construyendo df_all'):
    img_path = png_index.get(study_id)
    if img_path is None: continue  # estudio sin PNG procesado
    birads_max, density_char = '', ''
    for _, r in sdf.iterrows():
        d = safe_last_alnum(r.get('breast_density'))
        b = safe_last_alnum(r.get('breast_birads'))
        if not density_char and d: density_char = d.upper()
        if b.isdigit() and (not birads_max or int(b) > int(birads_max)): birads_max = b
    findings, mass, calc, asym = build_findings(finding_df, study_id)
    # Suspicion: mapeo BIRADS -> 3 clases para clasificacion de sospecha
    suspicion = 'healthy' if birads_max=='1' else ('benign' if birads_max in {'2','3'} else 'suspicious')
    records.append({
        'study_id'     : study_id,
        'image_path'   : img_path,
        'breast_birads': f'BI-RADS {birads_max}',
        'split'        : sdf['split'].iloc[0],
        'report'       : json.dumps({
            'density': density_char, 'findings': findings,
            'birads': birads_max, 'mass': mass,
            'calcification': calc, 'asymmetry': asym,
            'suspicion': suspicion,
        })
    })

df_all = pd.DataFrame(records)
print(f'\ndf_all total: {len(df_all)} estudios')
print(df_all['breast_birads'].value_counts().sort_index())

PNGs procesados encontrados: 4053


Construyendo df_all: 100%|██████████| 5000/5000 [00:10<00:00, 462.82it/s]


df_all total: 4053 estudios
breast_birads
BI-RADS 1    1962
BI-RADS 2    1174
BI-RADS 3     436
BI-RADS 4     368
BI-RADS 5     113
Name: count, dtype: int64


## 5. Split train/val + rebalanceo con augmentadas

**Problema:** VinDr-Mammo está muy desbalanceado — BIRADS 1/2 dominan (~70% del train). Sin rebalanceo el modelo aprende a predecir siempre 1 o 2.

**Estrategia de rebalanceo:**
- BIRADS 1, 2: downsample a 500 (suficientes originales disponibles)
- BIRADS 3, 4: upsample usando augmentadas hasta 500 (originales + flips generados)
- BIRADS 5: upsample usando augmentadas hasta 200 (originales + flips + rotaciones)

**aug_index:** mapea cada `study_id` a sus versiones augmentadas (`_flip`, `_rot`, `_rotx`).
Los estudios augmentados heredan el mismo `report` JSON que el original porque la augmentación no cambia el diagnóstico.

**df_balanced:** resultado final — 2200 filas de train perfectamente balanceadas.

In [5]:
import numpy as np
import re

# Separar train y val (test set de VinDr)
df_train_orig = df_all[df_all['split']=='training'].reset_index(drop=True)
val_df        = df_all[df_all['split']=='test'].reset_index(drop=True)

print(f'Train originales: {len(df_train_orig)}')
print(f'Val (test set):   {len(val_df)}')
print()

# --- Funcion para swap left<->right en findings ---
def swap_findings(report_json_str):
    """
    Para imagenes flipadas horizontalmente, invierte left<->right en el campo findings.
    El flip horizontal del merge 2x2 (R_CC|L_CC / R_MLO|L_MLO) produce
    (L_CC|R_CC / L_MLO|R_MLO), por lo que la mama que visualmente
    aparecia a la izquierda ahora aparece a la derecha y viceversa.
    """
    rep = json.loads(report_json_str)
    findings = rep.get('findings', '')
    if findings and findings != 'Healthy Breast. No Findings':
        # Swap usando placeholder para evitar doble reemplazo
        findings = re.sub(r'\bright\b', '__RIGHT__', findings, flags=re.IGNORECASE)
        findings = re.sub(r'\bleft\b', 'right', findings, flags=re.IGNORECASE)
        findings = findings.replace('__RIGHT__', 'left')
        rep['findings'] = findings
    return json.dumps(rep, ensure_ascii=False)

# --- Construir aug_index ---
aug_index = {}
for p in LOCAL_AUG_DIR.glob('*.png'):
    stem = p.stem
    study_id = re.sub(r'_(flip|rot|rotx)\d*$', '', stem)
    if study_id != stem:
        aug_index.setdefault(study_id, []).append(str(p))

print(f'Studies con augmentadas: {len(aug_index)}')
sample_sid = next(iter(aug_index))
print(f'Ejemplo: {sample_sid} -> {aug_index[sample_sid]}')
print()

# --- Rebalanceo ---
TARGET = {'BI-RADS 1':500, 'BI-RADS 2':500, 'BI-RADS 3':500, 'BI-RADS 4':500, 'BI-RADS 5':200}
balanced = []

for birads, target in TARGET.items():
    subset = df_train_orig[df_train_orig['breast_birads']==birads].copy()
    n = len(subset)
    print(f'{birads}: {n} originales -> target {target}')

    if n >= target:
        balanced.append(subset.sample(target, random_state=42))
        print(f'  -> Downsample a {target}')
    else:
        rows = [subset]
        needed = target - n

        aug_rows = []
        for _, row in subset.iterrows():
            sid = row['study_id']
            for aug_path in aug_index.get(sid, []):
                aug_row = row.copy()
                aug_row['image_path'] = aug_path
                # Si es flip, swap left<->right en findings del report
                if '_flip' in aug_path:
                    aug_row['report'] = swap_findings(row['report'])
                aug_rows.append(aug_row)

        if aug_rows:
            aug_df = pd.DataFrame(aug_rows)
            if len(aug_df) >= needed:
                rows.append(aug_df.sample(needed, random_state=42))
                print(f'  -> {n} orig + {needed} aug = {target}')
            else:
                rows.append(aug_df)
                still_needed = needed - len(aug_df)
                rows.append(subset.sample(still_needed, replace=True, random_state=42))
                print(f'  -> {n} orig + {len(aug_df)} aug + {still_needed} duplicados = {target}')
        else:
            rows.append(subset.sample(needed, replace=True, random_state=42))
            print(f'  -> {n} orig + {needed} duplicados = {target}')

        balanced.append(pd.concat(rows))

df_balanced = pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)

aug_count  = df_balanced['image_path'].str.contains('_flip|_rot').sum()
orig_count = len(df_balanced) - aug_count

print(f'\n=== df_balanced ===')
print(df_balanced['breast_birads'].value_counts().sort_index())
print(f'Total: {len(df_balanced)}')
print(f'Originales: {orig_count} | Augmentadas: {aug_count}')

for p in tqdm(val_df['image_path'].tolist(), desc='Cache val'):
    rp = resolve_path(p)
    if rp not in IMAGE_CACHE:
        try: IMAGE_CACHE[rp] = PILImage.open(rp).convert('RGB')
        except: pass
print(f'Cache total: {len(IMAGE_CACHE)}')

Train originales: 3053
Val (test set):   1000

Studies con augmentadas: 450
Ejemplo: 3ac71cf287f547a9e68a30a5620b7eb9 -> ['/content/processed_aug/3ac71cf287f547a9e68a30a5620b7eb9_flip.png']

BI-RADS 1: 1468 originales -> target 500
  -> Downsample a 500
BI-RADS 2: 855 originales -> target 500
  -> Downsample a 500
BI-RADS 3: 345 originales -> target 500
  -> 345 orig + 155 aug = 500
BI-RADS 4: 295 originales -> target 500
  -> 295 orig + 205 aug = 500
BI-RADS 5: 90 originales -> target 200
  -> 90 orig + 110 aug = 200

=== df_balanced ===
breast_birads
BI-RADS 1    500
BI-RADS 2    500
BI-RADS 3    500
BI-RADS 4    500
BI-RADS 5    200
Name: count, dtype: int64
Total: 2200
Originales: 1730 | Augmentadas: 470


Cache val: 100%|██████████| 1000/1000 [00:00<00:00, 64524.78it/s]

Cache total: 5253


## 6. CONFIG — hiperparámetros del fine-tuning

- : rango LoRA. Dimension del subespacio de adaptacion (AMRG + MammoWise: 32)
- : escala LoRA. **Run 3: alpha=r → scaling=1.0** (estandar LoRA paper). Run 1-2 usaban alpha=16 (scaling=0.5), lo que reducia la actualizacion efectiva en espacio de pesos a la mitad. DeltaW_eff = (alpha/r) * lr = 1.0 * 1e-4 = 1e-4 (antes: 5e-5)
- : learning rate. Con alpha=32 la actualizacion efectiva es equivalente a lr=2e-4 con alpha=16
- : overfitting confirmado despues de epoca 10 en Run 1 (acc BIRADS cayo post-ep12) y en MammoWise (0.6355 ep10 → 0.5139 ep15). save_total_limit=10 ahora guarda todos los checkpoints
- : batch efectivo = 1×8 = 8. Simula batch size 8 en una GPU con 1 imagen por step
- : muestras por clase en validacion generativa (costosa, se hace por epocas)

In [6]:
CONFIG = {
    'model_id'           : 'google/medgemma-4b-it',
    'lora_r'             : 32,       # Rango LoRA (AMRG: 32)
    'lora_alpha'         : 32,       # Escala LoRA, ratio alpha/r = 1.0 (Run 3: scaling estandar LoRA)
    'lora_dropout'       : 0.05,     # Dropout en adaptadores LoRA
    'lr'                 : 1e-4,     # Learning rate
    'epochs'             : 10,       # Epocas: overfitting confirmado post-epoca 10 (MammoWise + Run 1)
    'batch_size'         : 1,        # Batch size por GPU (limitado por VRAM)
    'grad_accum'         : 8,        # Pasos de acumulacion -> batch efectivo = 8
    'max_grad_norm'      : 1.0,      # Clip de gradiente para estabilidad
    'warmup_ratio'       : 0.05,     # 5% de steps para warmup lineal
    'max_new_tokens_val' : 124,       # Tokens maximos en generacion de validacion
    'val_n_per_class'    : 15,       # Muestras por clase en validacion generativa
    'seed'               : 20201531, # Semilla de reproducibilidad
}
print(CONFIG)

{'model_id': 'google/medgemma-4b-it', 'lora_r': 32, 'lora_alpha': 32, 'lora_dropout': 0.05, 'lr': 0.0001, 'epochs': 10, 'batch_size': 1, 'grad_accum': 8, 'max_grad_norm': 1.0, 'warmup_ratio': 0.05, 'max_new_tokens_val': 124, 'val_n_per_class': 15, 'seed': 20201531}


## 7. Prompt del sistema + targets + parsers

**SYSTEM_PROMPT:** rol del modelo (radiologo certificado, meticulos, reporte clínico).

**USER_PROMPT:** instrucciones completas incluyendo:
- Layout del merge 2x2 (top-left=L_CC, top-right=R_CC, bot-left=L_MLO, bot-right=R_MLO)
- Clasificación ACR de densidad (A/B/C/D)
- Formato de findings con lateralidad y vista
- Escala BI-RADS (1-5)
- Instrucción de output: SOLO JSON válido, sin markdown

**build_target():** extrae del report JSON solo los campos que el modelo debe predecir (density, findings, birads) — omite mass/calcification/asymmetry/suspicion del target de training.

**parse_birads / parse_density:** extractores robustos del output generado por el modelo. Manejan casos donde el modelo genera texto extra o formatos ligeramente distintos.

In [7]:
import torch, re, json

SYSTEM_PROMPT = (
    'You are a board-certified breast radiologist with extensive experience in screening '
    'mammography. You are meticulous and produce clear, clinically actionable reports.'
)

USER_PROMPT = (
    'Interpret the mammogram and produce a structured report.\n\n'
    'Image layout (2x2 composite):\n'
    '  top-left  = RIGHT breast, cranio-caudal (CC) view\n'
    '  top-right = LEFT breast, cranio-caudal (CC) view\n'
    '  bot-left  = RIGHT breast, medio-lateral oblique (MLO) view\n'
    '  bot-right = LEFT breast, medio-lateral oblique (MLO) view\n\n'
    'Breast density (ACR):\n'
    '  A: Almost entirely fatty\n'
    '  B: Scattered fibroglandular densities\n'
    '  C: Heterogeneously dense\n'
    '  D: Extremely dense\n\n'
    'Findings: describe abnormalities with laterality and view. '
    'Possible findings: mass, suspicious calcification, focal asymmetry, '
    'architectural distortion, asymmetry, suspicious lymph node, skin thickening, '
    'nipple retraction, global asymmetry, skin retraction. '
    'Format: "<finding> found in <left|right> <cranio-caudal (CC)|medio-lateral oblique (MLO)>". '
    'Multiple findings separated by " and ". '
    'If none: "Healthy Breast. No Findings".\n\n'
    'BI-RADS overall assessment:\n'
    '  1: Negative  2: Benign  3: Probably benign\n'
    '  4: Suspicious  5: Highly suggestive of malignancy\n\n'
    'Return ONLY valid JSON. No markdown. No explanation. No text outside the JSON.\n'
    '{"birads": "<1 or 2 or 3 or 4 or 5>", "density": "<A or B or C or D>", "findings": "<description>"}'
)

def build_target(row):
    """Construye el JSON target que el modelo debe generar para una fila del dataset."""
    rep = json.loads(row['report'])
    return json.dumps({
        'birads'  : rep['birads'],
        'density' : rep['density'],
        'findings': rep['findings'],
    }, ensure_ascii=False)

def parse_birads(text):
    """Extrae BIRADS (1-5) del output generado por el modelo. Robusto a formatos variables."""
    try:
        d = json.loads(text)
        m = re.search(r'([1-5])', str(d.get('birads','')))
        if m: return m.group(1)
    except: pass
    m = re.search(r'"birads"\s*:\s*"?([1-5])', text)
    return m.group(1) if m else 'UNKNOWN'

def parse_density(text):
    """Extrae density (A/B/C/D) del output generado por el modelo."""
    try:
        d = json.loads(text)
        v = str(d.get('density','')).strip().upper()
        for l in ['A','B','C','D']:
            if l in v: return l
    except: pass
    m = re.search(r'"density"\s*:\s*"?([A-D])', text)
    return m.group(1) if m else 'UNKNOWN'

print('Parsers OK')
print('\n5 ejemplos de targets (verificacion):')
for i in [0, 200, 500, 1000, 1500]:
    if i < len(df_balanced):
        row = df_balanced.iloc[i]
        print(f'  {row["breast_birads"]} | {build_target(row)}')

Parsers OK

5 ejemplos de targets (verificacion):
  BI-RADS 3 | {"birads": "3", "density": "C", "findings": "mass found in left cranio-caudal (CC) and mass found in left medio-lateral oblique (MLO)"}
  BI-RADS 3 | {"birads": "3", "density": "C", "findings": "mass found in right medio-lateral oblique (MLO)"}
  BI-RADS 4 | {"birads": "4", "density": "C", "findings": "mass found in left medio-lateral oblique (MLO) and mass found in left cranio-caudal (CC)"}
  BI-RADS 1 | {"birads": "1", "density": "C", "findings": "Healthy Breast. No Findings"}
  BI-RADS 2 | {"birads": "2", "density": "C", "findings": "Healthy Breast. No Findings"}


## 8. Login HuggingFace + Processor

MedGemma es un modelo de acceso restringido — requiere login con token HF que tenga acceso aprobado a `google/medgemma-4b-it`.

El `AutoProcessor` maneja tanto el texto (tokenizer) como las imágenes (SigLIP preprocessing). `padding_side='right'` es requerido durante training (para que el padding no interfiera con el loss masking).

In [9]:
from huggingface_hub import login
from transformers import AutoProcessor
from google.colab import userdata

# Login con token HF almacenado en Colab Secrets (no hardcodear el token)
login(token=userdata.get('tesis'))

processor = AutoProcessor.from_pretrained(CONFIG['model_id'], use_fast=True)
processor.tokenizer.padding_side = 'right'  # requerido para CLM training

# Detectar el ID del token de imagen especial de MedGemma
# Necesario para maskear esos tokens en el calculo de loss
IMAGE_TOKEN_ID = None
for tok in ['<image_soft_token>','<img>','<image>']:
    tid = processor.tokenizer.convert_tokens_to_ids(tok)
    if tid and tid != processor.tokenizer.unk_token_id:
        IMAGE_TOKEN_ID = tid; break
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = 262144  # fallback conocido para MedGemma

print(f'IMAGE_TOKEN_ID: {IMAGE_TOKEN_ID}')
print('Processor OK')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

IMAGE_TOKEN_ID: 262144
Processor OK


## 9. Dataset + Stratified Batch Sampler

**VinDrDataset:** PyTorch Dataset que retorna imagen PIL + mensajes en formato chat de MedGemma.

**StratifiedBatchSampler:** garantiza que cada batch efectivo (grad_accum pasos) contenga ejemplos de TODAS las 5 clases de BIRADS.

**Por qué es necesario:** VinDr tiene correlación casi perfecta findings→BIRADS en el texto sintético. Sin sampler, batches consecutivos dominados por BIRADS 1/2 refuerzan el prior del modelo. El sampler forza al modelo a discriminar entre las 5 clases desde el step 1, previniendo colapso a las clases mayoritarias.

**Implementación:** permuta indices por clase al inicio de cada época, luego toma `n_per_class` de cada clase por batch. `n_per_class = batch_size // n_classes = 8//5 = 1`.

In [10]:
from torch.utils.data import Dataset, Sampler
import numpy as np

class VinDrDataset(Dataset):
    """Dataset de VinDr-Mammo para fine-tuning de MedGemma."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = resolve_path(row['image_path'])
        # Usar cache RAM si disponible, sino cargar desde disco
        img      = IMAGE_CACHE.get(img_path) or PILImage.open(img_path).convert('RGB')
        target   = build_target(row)
        # Formato de mensajes chat: usuario (prompt+imagen) -> asistente (JSON target)
        messages = [
            {'role':'user','content':[
                {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
                {'type':'image','image': img},
            ]},
            {'role':'assistant','content':[{'type':'text','text': target}]},
        ]
        return {'messages': messages, 'image': img, 'target_text': target}

class StratifiedBatchSampler(Sampler):
    """
    Sampler estratificado que garantiza representacion de todas las clases
    en cada batch efectivo. Cada batch contiene n_per_class ejemplos por clase.
    Numero de batches limitado por la clase con menos ejemplos.
    """
    def __init__(self, labels, batch_size, seed=42):
        self.labels      = np.array(labels)
        self.classes     = np.unique(self.labels)
        self.n_per_class = max(1, batch_size // len(self.classes))
        self.seed        = seed

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        # Permutacion aleatoria de indices por clase al inicio de cada epoca
        class_idx = {
            c: rng.permutation(np.where(self.labels==c)[0]).tolist()
            for c in self.classes
        }
        ptrs = {c: 0 for c in self.classes}
        # Numero de batches limitado por la clase minoritaria
        n_batches = min(len(idx)//self.n_per_class for idx in class_idx.values())
        for _ in range(n_batches):
            batch = []
            for c in self.classes:
                p = ptrs[c]
                batch.extend(class_idx[c][p:p+self.n_per_class])
                ptrs[c] += self.n_per_class
            rng.shuffle(batch)  # mezclar clases dentro del batch
            yield batch

    def __len__(self):
        return min((self.labels==c).sum()//self.n_per_class for c in self.classes)

# Construir datasets
birads_labels = df_balanced['breast_birads'].str.extract(r'(\d)')[0].astype(int).values
batch_eff     = CONFIG['batch_size'] * CONFIG['grad_accum']  # batch efectivo = 8

train_ds = VinDrDataset(df_balanced)
val_ds   = VinDrDataset(
    val_df.groupby('breast_birads', group_keys=False).apply(
        lambda x: x.sample(min(CONFIG['val_n_per_class'], len(x)), random_state=42),
        include_groups=False
    ).reset_index(drop=True)
)
sampler = StratifiedBatchSampler(birads_labels, batch_size=batch_eff, seed=CONFIG['seed'])

print(f'Train dataset: {len(train_ds)} muestras')
print(f'Val dataset:   {len(val_ds)} muestras ({CONFIG["val_n_per_class"]} por clase)')
print(f'Batches/epoca: {len(sampler)} | batch efectivo: {batch_eff}')
print(f'n_per_class por batch: {sampler.n_per_class}')
print('\nVerificacion primeros 3 batches (clases representadas):')
for i, b in enumerate(sampler):
    if i >= 3: break
    print(f'  Batch {i}: clases {sorted([birads_labels[j] for j in b])}')

Train dataset: 2200 muestras
Val dataset:   75 muestras (15 por clase)
Batches/epoca: 200 | batch efectivo: 8
n_per_class por batch: 1

Verificacion primeros 3 batches (clases representadas):
  Batch 0: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Batch 1: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Batch 2: clases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 10. Collate — preparacion de batches + masking de loss

**Funcion collate_fn:** convierte una lista de ejemplos en un batch tokenizado listo para el modelo.

**Masking de loss (labels=-100):** el modelo solo aprende de los tokens de la respuesta del asistente. Se enmascaran:
1. Tokens de padding
2. Tokens de imagen (`IMAGE_TOKEN_ID`)
3. Token especial BOI (begin-of-image) si existe
4. Todo el prompt del usuario (hasta `<start_of_turn>model`)

**Sin truncación:** los inputs de MedGemma con imagen 896x896 tienen ~700 tokens (imagen=256 tokens + prompt~400 + respuesta~50). Caben en el contexto de 8192 tokens del modelo sin necesidad de truncar.

In [11]:
from functools import partial

def collate_fn(examples, processor, image_token_id):
    """
    Prepara un batch para MedGemma:
    1. Aplica chat template a los mensajes
    2. Tokeniza texto + procesa imagenes
    3. Crea labels maskeando todo excepto la respuesta del asistente
    """
    texts, images = [], []
    for ex in examples:
        images.append([ex['image']])
        # apply_chat_template formatea los mensajes en el formato de MedGemma
        chat = processor.apply_chat_template(
            ex['messages'], add_generation_prompt=False, tokenize=False).strip()
        texts.append(chat)

    # Sin truncacion — inputs ~700 tokens caben bien en el contexto
    batch = processor(text=texts, images=images, return_tensors='pt', padding='longest')
    labels = batch['input_ids'].clone()

    # 1. Maskear tokens de padding
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None: labels[labels==pad_id] = -100

    # 2. Maskear tokens de imagen (no contribuyen al loss de texto)
    if image_token_id is not None: labels[labels==image_token_id] = -100

    # 3. Maskear token BOI (begin-of-image) si existe
    boi = processor.tokenizer.special_tokens_map.get('boi_token')
    if boi:
        bid = processor.tokenizer.convert_tokens_to_ids(boi)
        if bid and bid != processor.tokenizer.unk_token_id:
            labels[labels==bid] = -100

    # 4. Maskear el prompt completo del usuario (todo antes de <start_of_turn>model)
    # Solo los tokens de la respuesta del asistente contribuyen al loss
    model_turn_ids = processor.tokenizer.encode('<start_of_turn>model', add_special_tokens=False)
    n = len(model_turn_ids)
    for i in range(labels.shape[0]):
        seq = batch['input_ids'][i].tolist()
        found = None
        for k in range(len(seq)-n):
            if seq[k:k+n] == model_turn_ids: found = k+n
        if found is not None: labels[i,:found] = -100

    batch['labels'] = labels
    return batch

collate = partial(collate_fn, processor=processor, image_token_id=IMAGE_TOKEN_ID)

# Sanity check: verificar que el masking funciona correctamente
b = collate([train_ds[0], train_ds[1]])
n_active = (b['labels'] != -100).sum().item()
print(f'Tokens activos en loss (2 muestras): {n_active} (~{n_active//2} por muestra)')
print(f'Shape input_ids: {b["input_ids"].shape}')
print('Tokens activos decodificados (debe ser solo el JSON target):')
for i in range(2):
    active = b['input_ids'][i][b['labels'][i] != -100]
    decoded = processor.tokenizer.decode(active)
    print(f'  Muestra {i}: {repr(decoded)}')
print('Collate OK')

Tokens activos en loss (2 muestras): 93 (~46 por muestra)
Shape input_ids: torch.Size([2, 661])
Tokens activos decodificados (debe ser solo el JSON target):
  Muestra 0: '\n{"birads": "3", "density": "C", "findings": "mass found in left cranio-caudal (CC) and mass found in left medio-lateral oblique (MLO)"}<end_of_turn>'
  Muestra 1: '\n{"birads": "3", "density": "C", "findings": "focal asymmetry found in right cranio-caudal (CC) and focal asymmetry found in right medio-lateral oblique (MLO)"}<end_of_turn>'
Collate OK


## 11. Cargar modelo + LoRA

**Cuantizacion 4-bit NF4:** reduce MedGemma 4B de ~16GB a ~5GB VRAM. `bnb_4bit_compute_dtype=bfloat16` mantiene precision de calculo.

**LoRA target modules:** siguiendo AMRG, se aplica LoRA a TODOS los linear layers del LM (attention q/k/v/o + FFN gate/up/down), excluyendo el vision encoder (SigLIP). Esto da ~730M parametros entrenables (~22% del total).

**modules_to_save=['lm_head', 'embed_tokens'] + ensure_weight_tying=True:** Run 4: Gemma tie lm_head.weight == embed_tokens.weight — PEFT silently breaks this tie when solo lm_head esta en modules_to_save (PEFT issue #2864). Ambos deben entrenarse juntos para mantener coherencia en generacion.

**align_dtypes:** alinea todos los parametros entrenables a bfloat16 para evitar errores de tipo en el backward pass con 4-bit quantization.

In [12]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import re as _re

torch_dtype = torch.bfloat16

# Configuracion de cuantizacion 4-bit NF4 (QLoRA)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # Normal Float 4 - mejor para pesos pre-entrenados
    bnb_4bit_compute_dtype=torch_dtype,   # bfloat16 para calculos
    bnb_4bit_use_double_quant=True,       # doble cuantizacion para mas ahorro
)
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Cargar modelo base cuantizado
model = AutoModelForImageTextToText.from_pretrained(
    CONFIG['model_id'],
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    attn_implementation='eager',  # 'eager' requerido para gradient checkpointing con 4-bit
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False    # incompatible con gradient checkpointing
model.enable_input_require_grads()  # necesario para LoRA con gradient checkpointing

# Seleccionar modulos target para LoRA: todos los linear del LM, excluyendo vision encoder
# AMRG: LoRA en attention (q/k/v/o) + FFN (gate/up/down) del language model
lm_modules = [
    n for n,_ in model.named_modules()
    if _re.match(
        r'^(?!.*vision).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$', n
    )
]
print(f'LoRA target modules: {len(lm_modules)} capas')

# Configuracion LoRA
peft_cfg = LoraConfig(
    r               = CONFIG['lora_r'],
    lora_alpha      = CONFIG['lora_alpha'],
    lora_dropout    = CONFIG['lora_dropout'],
    bias            = 'none',
    target_modules  = lm_modules,
    task_type       = 'CAUSAL_LM',
    modules_to_save     = ['lm_head', 'embed_tokens'],  # Run 4: fix weight tying (PEFT #2864)
    ensure_weight_tying = True,                          # Gemma: lm_head.weight == embed_tokens.weight
)
model = get_peft_model(model, peft_cfg)

def align_dtypes(model, dtype=torch.bfloat16):
    """Alinea parametros entrenables a dtype para evitar errores con 4-bit quantization."""
    n = 0
    for p in model.parameters():
        if p.requires_grad and p.dtype in (torch.float32, torch.float16):
            p.data = p.data.to(dtype)
            n += 1
    print(f'Aligned {n} params -> {dtype}')

align_dtypes(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parametros entrenables: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f}GB')

GPU: NVIDIA A100-SXM4-40GB


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

LoRA target modules: 238 capas
Aligned 476 params -> torch.bfloat16
Parametros entrenables: 1,402,109,952 (36.02%)
VRAM usada: 6.0GB


## 12. Funcion de validacion generativa

**Por qué validacion generativa en vez de val_loss:** la loss de validacion mide perplexity del texto generado, no accuracy diagnostica. Un modelo puede tener baja loss pero predecir siempre el mismo BIRADS.

**validate_custom:** genera respuestas reales con `model.generate()` y extrae BIRADS+density con los parsers. Reporta accuracy y F1 macro por clase.

**temperature=0.1 (AMRG):** generacion casi determinista para evaluacion reproducible. `do_sample=True` con temperatura muy baja es equivalente a greedy pero mas estable.

**Detalles de implementacion:** se desactiva gradient checkpointing y se activa `use_cache=True` durante generacion (requerido para `model.generate()`), luego se restaura para training.

In [13]:
from sklearn.metrics import f1_score

device = torch.device('cuda')

def validate_custom(model, val_df, processor, device,
                    n_per_class=15, max_new_tokens=124, show_examples=5):
    """
    Validacion generativa: genera respuestas reales y mide accuracy/F1 de BIRADS y density.
    Mas costosa que val_loss pero refleja el rendimiento diagnostico real.
    """
    # Preparar modelo para inferencia
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    orig_pad = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = 'left'  # requerido para generacion

    # Muestra estratificada del val set
    val_small = val_df.groupby('breast_birads', group_keys=False).apply(
        lambda x: x.sample(min(n_per_class, len(x)), random_state=42),
        include_groups=False
    ).reset_index(drop=True)

    preds, refs, dens_preds, dens_refs = [], [], [], []
    for idx, (_, row) in enumerate(val_small.iterrows()):
        img_path = resolve_path(row['image_path'])
        img = IMAGE_CACHE.get(img_path) or PILImage.open(img_path).convert('RGB')
        msgs = [{'role':'user','content':[
            {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
            {'type':'image','image': img}]}]
        text   = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=[img], return_tensors='pt').to(device)
        inputs = {k: v.to(torch.bfloat16) if torch.is_floating_point(v) else v
                  for k,v in inputs.items()}
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                gen = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
        # Decodificar solo los tokens nuevos (excluir el prompt)
        dec = processor.tokenizer.decode(
            gen[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        del inputs, gen; torch.cuda.empty_cache()

        rep   = json.loads(row['report'])
        p_bir = parse_birads(dec)
        p_den = parse_density(dec)

        if idx < show_examples:
            print(f"\n{'='*50}")
            print(f"GT birads={rep['birads']} density={rep['density']}")
            print(f"OUTPUT: {repr(dec)}")
            print(f"Parsed -> birads={p_bir} {'OK' if p_bir==rep['birads'] else 'FAIL'} | "
                  f"density={p_den} {'OK' if p_den==rep['density'] else 'FAIL'}")

        preds.append(p_bir);      refs.append(str(rep['birads']))
        dens_preds.append(p_den); dens_refs.append(rep['density'])

    # Restaurar modelo para training
    processor.tokenizer.padding_side = orig_pad
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.train()

    # Metricas globales
    n     = len(refs)
    b_acc = sum(p==r for p,r in zip(preds,refs))/n
    d_acc = sum(p==r for p,r in zip(dens_preds,dens_refs))/n
    b_f1  = f1_score(refs, preds, average='macro', zero_division=0)
    d_f1  = f1_score(dens_refs, dens_preds, average='macro', zero_division=0)

    print(f'\n  BI-RADS acc={b_acc:.4f} F1={b_f1:.4f} | Density acc={d_acc:.4f} F1={d_f1:.4f}')
    for c in ['1','2','3','4','5']:
        rs = [r for r in refs if r==c]
        if rs:
            ps = [p for p,r in zip(preds,refs) if r==c]
            print(f'    BI-RADS {c}: acc={sum(pp==rr for pp,rr in zip(ps,rs))/len(rs):.4f} n={len(rs)}')

    return {'birads_accuracy':b_acc,'birads_f1':b_f1,
            'density_accuracy':d_acc,'density_f1':d_f1,'n_total':n}

print('validate_custom OK')

validate_custom OK


## 13. Callbacks — monitoreo de colapso y loss por step

**CollapseCallback:** evalua cada `check_every` steps si el modelo esta colapsando (prediciendo siempre la misma clase). Detecta colapso cuando una clase domina >60% de las predicciones. Imprime distribucion de predicciones y accuracy por clase.

**StepLossCallback:** imprime loss, learning rate y grad_norm en cada logging step. Util para monitorear que el training converge normalmente.

**Por que no EarlyStopping:** la loss de validacion no correlaciona con accuracy diagnostica en este task. Se entrena el maximo de epocas y se selecciona el mejor checkpoint offline via validate_custom.

In [14]:
from transformers import TrainerCallback
from collections import Counter

class CollapseCallback(TrainerCallback):
    """
    Detecta colapso de predicciones durante training.
    Evalua cada check_every steps sobre un subset fijo de val.
    Alerta si >60% de predicciones caen en una sola clase.
    Muestra el output crudo del modelo para diagnostico visual.
    """
    def __init__(self, val_df, processor, device, check_every=200):
        self.val_fixed = val_df.groupby('breast_birads').apply(
            lambda x: x.sample(min(3,len(x)), random_state=42), include_groups=False
        ).reset_index(drop=True)
        self.processor   = processor
        self.device      = device
        self.check_every = check_every
        self.history     = []

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.check_every != 0 or state.global_step == 0: return
        model.eval(); model.gradient_checkpointing_disable()
        model.config.use_cache = True
        orig = self.processor.tokenizer.padding_side
        self.processor.tokenizer.padding_side = 'left'
        preds, refs, outputs = [], [], []
        with torch.no_grad():
            for _, row in self.val_fixed.iterrows():
                try:
                    img = IMAGE_CACHE.get(resolve_path(row['image_path'])) or \
                          PILImage.open(resolve_path(row['image_path'])).convert('RGB')
                    msgs = [{'role':'user','content':[
                        {'type':'text','text': SYSTEM_PROMPT+'\n\n'+USER_PROMPT},
                        {'type':'image','image': img}]}]
                    text = self.processor.apply_chat_template(
                        msgs, add_generation_prompt=True, tokenize=False)
                    inputs = self.processor(text=text, images=[img],
                                           return_tensors='pt')
                    # Cast agresivo: device + dtype correcto para cada tensor
                    inputs = {k: v.to(device=self.device, dtype=torch.bfloat16)
                              if torch.is_floating_point(v)
                              else v.to(device=self.device)
                              for k,v in inputs.items()}
                    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                        gen = model.generate(**inputs, max_new_tokens=124, do_sample=False)
                    dec = self.processor.tokenizer.decode(
                        gen[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
                    rep = json.loads(row['report'])
                    preds.append(parse_birads(dec))
                    refs.append(str(rep['birads']))
                    outputs.append({
                        'gt_birads' : rep['birads'],
                        'gt_density': rep['density'],
                        'raw'       : dec
                    })
                    del inputs, gen
                except Exception as e:
                    preds.append('UNKNOWN'); refs.append('?')
                    outputs.append({'gt_birads':'?','gt_density':'?','raw':f'ERROR: {e}'})

        torch.cuda.empty_cache()
        self.processor.tokenizer.padding_side = orig
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.train()

        c = Counter(preds)
        dom, cnt = c.most_common(1)[0]
        pct    = cnt/len(preds)
        status = f'COLAPSO -> "{dom}" {pct:.0%}' if pct>0.6 else 'OK'
        per_class = {}
        for b in ['1','2','3','4','5']:
            gt_idx = [i for i,r in enumerate(refs) if r==b]
            if gt_idx:
                correct = sum(1 for i in gt_idx if preds[i]==b)
                per_class[b] = f'{correct}/{len(gt_idx)}'

        print(f'\n  [step {state.global_step}] dist={dict(c)} | {status}')
        print(f'  Acc por clase: {per_class}')
        print(f'  --- Outputs (primeros 5) ---')
        for o, p in list(zip(outputs, preds))[:5]:
            marker = 'OK' if p == o['gt_birads'] else 'FAIL'
            print(f'  GT birads={o["gt_birads"]} density={o["gt_density"]}')
            print(f'  PRED: {o["raw"][:200]}')
            print(f'  -> birads parsed={p} {marker}')
            print()

        self.history.append({'step':state.global_step,'dist':dict(c),'per_class':per_class})


class StepLossCallback(TrainerCallback):
    """Loguea loss, learning rate y grad_norm en cada logging step."""
    def __init__(self): self.log = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            e = {'step':state.global_step,'loss':logs.get('loss'),
                 'lr':logs.get('learning_rate'),'grad_norm':logs.get('grad_norm','N/A')}
            self.log.append(e)
            gn = e['grad_norm'] if isinstance(e['grad_norm'],str) else f"{e['grad_norm']:.2f}"
            print(f"  [step {e['step']}] loss={e['loss']:.4f} lr={e['lr']:.2e} grad={gn}")

print('Callbacks OK')

Callbacks OK


In [ ]:
# --- Celda de prueba del CollapseCallback ---
print('Probando CollapseCallback...')
test_cb = CollapseCallback(val_df, processor, device, check_every=1)

class FakeState:
    global_step = 1

class FakeArgs:
    pass

test_cb.on_step_end(FakeArgs(), FakeState(), None, model=model)
print('Prueba OK')

Probando CollapseCallback...

  [step 1] dist={'4': 12, '1': 1, 'UNKNOWN': 2} | COLAPSO -> "4" 80%
  Acc por clase: {'1': '0/3', '2': '0/3', '3': '0/3', '4': '3/3', '5': '0/3'}
  --- Outputs (primeros 5) ---
  GT birads=1 density=C
  PRED: ```json
{"density": "B", "findings": "A 0.8 cm, irregular, slightly spiculated density is seen in the right breast, CC view. No findings are seen in the left breast, CC view. A 0.6 cm, irregular, slig
  -> birads parsed=4 FAIL

  GT birads=1 density=D
  PRED: ```json
{"density": "B", "findings": "A 1.0 cm mass is seen in the right breast, CC view. A 0.8 cm mass is seen in the left breast, MLO view.", "birads": "4"}
```
  -> birads parsed=4 FAIL

  GT birads=1 density=C
  PRED: ```json
{
  "density": "B",
  "findings": "A 0.8 cm, slightly irregular, non-mass-like density is seen in the right breast, CC view. A 0.6 cm, slightly irregular, non-mass-like density is seen in the 
  -> birads parsed=4 FAIL

  GT birads=2 density=C
  PRED: ```json
{"density"

## 14. Baseline sin fine-tuning

Evaluar el modelo base ANTES de cualquier fine-tuning para establecer el punto de partida.
MedGemma base sin fine-tuning en VinDr tipicamente logra ~20% acc en BIRADS y ~13% en density.
Esto confirma que el formato 2x2 y el dominio especifico requieren fine-tuning.

In [ ]:
print('=== BASELINE SIN FINE-TUNING ===')
print('(esperado: ~20% BIRADS acc, ~13% density acc)')
baseline_results = validate_custom(model, val_df, processor, device,
                                   n_per_class=3, max_new_tokens=124, show_examples=5)

=== BASELINE SIN FINE-TUNING ===
(esperado: ~20% BIRADS acc, ~13% density acc)

GT birads=1 density=C
OUTPUT: '```json\n{"density": "C", "findings": "Multiple clustered microcalcifications found in the right breast, CC view. Multiple clustered microcalcifications found in the left breast, CC view. Architectural distortion is present in both breasts. The right breast shows a possible asymmetry in the upper outer quadrant. The left breast shows a possible asymmetry in the upper outer quadrant.", "birads": "4"}\n```'
Parsed -> birads=4 FAIL | density=C OK

GT birads=1 density=C
OUTPUT: '```json\n{"density": "B", "findings": "A 1.0 cm mass is seen in the right breast, CC view. A 0.8 cm suspicious calcification is seen in the right breast, CC view. A 0.6 cm suspicious calcification is seen in the left breast, CC view. Architectural distortion is seen in the right breast, MLO view.", "birads": "4"}\n```'
Parsed -> birads=4 FAIL | density=B FAIL

GT birads=1 density=C
OUTPUT: '```json\n{"dens

KeyboardInterrupt: 

## 15. Training principal

**SFTTrainer con batch sampler custom:** el `StratifiedBatchSampler` se pasa directamente al `DataLoader` del trainer via `train_dataset` + sampler. SFT (Supervised Fine-Tuning) con CLM loss estandar — sin class weighting (AMRG).

**Sin EarlyStopping:** se entrena el total de epocas. La decision de cual checkpoint usar se hace offline evaluando cada uno con `validate_custom`. La loss de val no es metrica fiable para BIRADS.

**Checkpoints cada epoca:** se guardan al final de cada epoca (~200 steps) para poder evaluar todos offline y elegir el mejor.

**Duracion estimada:** ~40min/epoca en A100 = ~7h para 10 epocas.

In [ ]:
import gc
# Limpiar VRAM antes de training
torch.cuda.empty_cache()
gc.collect()
model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
print(f'VRAM antes de training: {torch.cuda.memory_allocated()/1e9:.1f}GB')
print('Listo para training')

VRAM antes de training: 4.8GB
Listo para training


In [15]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
from trl import SFTConfig, SFTTrainer

step_cb     = StepLossCallback()
collapse_cb = CollapseCallback(val_df, processor, device, check_every=200)

sft_cfg = SFTConfig(
    output_dir                    = str(CHECKPOINT_DIR / 'run'),
    num_train_epochs              = CONFIG['epochs'],
    per_device_train_batch_size   = CONFIG['batch_size'],
    per_device_eval_batch_size    = 1,
    gradient_accumulation_steps   = CONFIG['grad_accum'],
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    optim                         = 'adamw_bnb_8bit',
    bf16                          = True,
    fp16                          = False,
    logging_steps                 = 25,
    eval_strategy                 = 'no',
    save_strategy                 = 'epoch',
    save_total_limit              = 10,
    load_best_model_at_end        = False,
    learning_rate                 = CONFIG['lr'],
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = CONFIG['warmup_ratio'],
    max_grad_norm                 = CONFIG['max_grad_norm'],
    report_to                     = 'none',
    dataset_kwargs                = {'skip_prepare_dataset': True},
    remove_unused_columns         = False,
    label_names                   = ['labels'],
    seed                          = CONFIG['seed'],
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_cfg,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    processing_class = processor,
    data_collator    = collate,
    callbacks        = [step_cb, collapse_cb],
)

# steps_per_epoch calculado desde el dataset balanceado (sin sampler estratificado)
steps_per_epoch = len(train_ds) // (CONFIG['batch_size'] * CONFIG['grad_accum'])
print(f'Steps/epoca: {steps_per_epoch}')
print(f'Total steps: ~{steps_per_epoch * CONFIG["epochs"]}')
print(f'LR={CONFIG["lr"]} | LoRA r={CONFIG["lora_r"]} alpha={CONFIG["lora_alpha"]} | epochs={CONFIG["epochs"]}')

trainer.train()
print('\nEntrenamiento completado')

import json as _j
out_dir = CHECKPOINT_DIR / 'run'
with open(out_dir / 'step_logs.json', 'w') as f:
    _j.dump(step_cb.log, f, indent=2)
with open(out_dir / 'collapse_logs.json', 'w') as f:
    _j.dump(collapse_cb.history, f, indent=2)
print(f'Logs guardados en {out_dir}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Steps/epoca: 275
Total steps: ~2750
LR=0.0001 | LoRA r=32 alpha=32 | epochs=10


Step,Training Loss
25,6.394700
50,1.987900
75,1.131800
100,1.023700
125,1.032200
150,1.022500
175,0.984400
200,0.929500
225,0.973800
250,0.945300


  [step 25] loss=6.3947 lr=1.74e-05 grad=41.00
  [step 50] loss=1.9879 lr=3.55e-05 grad=16.62
  [step 75] loss=1.1318 lr=5.36e-05 grad=17.50
  [step 100] loss=1.0237 lr=7.17e-05 grad=16.75
  [step 125] loss=1.0322 lr=8.99e-05 grad=16.75
  [step 150] loss=1.0225 lr=1.00e-04 grad=15.19
  [step 175] loss=0.9844 lr=1.00e-04 grad=9.81

  [step 200] dist={'1': 15} | COLAPSO -> "1" 100%
  Acc por clase: {'1': '3/3', '2': '0/3', '3': '0/3', '4': '0/3', '5': '0/3'}
  --- Outputs (primeros 5) ---
  GT birads=1 density=C
  PRED: {"birads": "1", "density": "C", "findings": "Healthy Breast. No Findings"}
  -> birads parsed=1 OK

  GT birads=1 density=C
  PRED: {"birads": "1", "density": "C", "findings": "Healthy Breast. No Findings"}
  -> birads parsed=1 OK

  GT birads=1 density=C
  PRED: {"birads": "1", "density": "C", "findings": "Healthy Breast. No Findings"}
  -> birads parsed=1 OK

  GT birads=2 density=B
  PRED: {"birads": "1", "density": "C", "findings": "Healthy Breast. No Findings"}
  -> 

In [24]:
import gc, json, torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
import pandas as pd, os

# ── CAMBIA SOLO ESTO ─────────────────────────────────────────────────────────
CKPT_PATH      = "/content/drive/MyDrive/vindr_1000/checkpoints/multitask_vf/run/checkpoint-2750"
BASE_MODEL     = "google/medgemma-4b-it"
MAX_NEW_TOKENS = 128
# ─────────────────────────────────────────────────────────────────────────────

# Limpieza VRAM
try:
    del eval_model, eval_proc
    torch.cuda.empty_cache(); gc.collect()
except: pass

# Procesador
eval_proc = AutoProcessor.from_pretrained(BASE_MODEL)
eval_proc.tokenizer.padding_side = "left"

# Modelo base + adapter
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
_base = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="eager",
)
eval_model = PeftModel.from_pretrained(_base, CKPT_PATH)
eval_model.eval()
del _base

# ── Parsers ───────────────────────────────────────────────────────────────────
def parse_field(text, field):
    try:
        start = text.index("{"); end = text.rindex("}") + 1
        return str(json.loads(text[start:end]).get(field, "")).strip()
    except:
        return ""

# ── Inferencia ────────────────────────────────────────────────────────────────
def infer_row(row):
    image = Image.open(row["image_path"]).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text",  "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "image", "image": image},
                                       {"type": "text",  "text": USER_PROMPT}]},
    ]
    inputs = eval_proc.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True,
    ).to(eval_model.device)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        out = eval_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    text = eval_proc.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    gt_birads  = row["breast_birads"].replace("BI-RADS ", "").strip()
    gt_density = json.loads(row["report"]).get("density", "").strip().upper()

    return gt_birads, parse_field(text, "birads"), gt_density, parse_field(text, "density").upper()

# ── Loop ──────────────────────────────────────────────────────────────────────
gts, preds         = [], []
density_gts, density_preds = [], []
parse_fails        = 0

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=CKPT_PATH.split("/")[-1]):
    try:
        gt, pred, gt_d, pred_d = infer_row(row)
    except Exception as e:
        gt   = row["breast_birads"].replace("BI-RADS ", "").strip()
        pred, gt_d, pred_d = "", "", ""

    if not pred:
        parse_fails += 1
        pred = "0"
    if not pred_d:
        pred_d = "X"

    gts.append(gt);       preds.append(pred)
    density_gts.append(gt_d); density_preds.append(pred_d)

# ── Métricas BI-RADS ─────────────────────────────────────────────────────────
b_labels = ["1","2","3","4","5"]
b_acc = sum(g == p for g, p in zip(gts, preds)) / len(gts)
b_f1  = f1_score(gts, preds, labels=b_labels, average="macro", zero_division=0)

print(f"\n{'='*50}")
print(f"Checkpoint : {CKPT_PATH.split('/')[-1]}")
print(f"\n── BI-RADS ──────────────────────────────────────")
print(f"Accuracy : {b_acc*100:.1f}%  ({sum(g==p for g,p in zip(gts,preds))}/{len(gts)})")
print(f"F1-macro : {b_f1:.4f}")
print(f"Parse fails: {parse_fails}")
print(classification_report(gts, preds, labels=b_labels, zero_division=0))

# ── Métricas Density ─────────────────────────────────────────────────────────
d_labels = ["A","B","C","D"]
d_acc = sum(g == p for g, p in zip(density_gts, density_preds)) / len(density_gts)
d_f1  = f1_score(density_gts, density_preds, labels=d_labels, average="macro", zero_division=0)

print(f"── Density ──────────────────────────────────────")
print(f"Accuracy : {d_acc*100:.1f}%  ({sum(g==p for g,p in zip(density_gts,density_preds))}/{len(density_gts)})")
print(f"F1-macro : {d_f1:.4f}")
print(classification_report(density_gts, density_preds, labels=d_labels, zero_division=0))
print(f"{'='*50}")

# ── Guardar ───────────────────────────────────────────────────────────────────
ckpt_name = CKPT_PATH.split("/")[-1]
out_dir   = "/content/drive/MyDrive/vindr_1000/eval_results"
os.makedirs(out_dir, exist_ok=True)

results_df = val_df[["study_id", "breast_birads"]].copy()
results_df["gt_birads"]    = gts
results_df["pred_birads"]  = preds
results_df["ok_birads"]    = results_df["gt_birads"] == results_df["pred_birads"]
results_df["gt_density"]   = density_gts
results_df["pred_density"] = density_preds
results_df["ok_density"]   = results_df["gt_density"] == results_df["pred_density"]
results_df.to_csv(f"{out_dir}/{ckpt_name}_preds.csv", index=False)

metrics = {
    "checkpoint":       ckpt_name,
    "birads_accuracy":  round(b_acc, 4),
    "birads_f1_macro":  round(b_f1, 4),
    "density_accuracy": round(d_acc, 4),
    "density_f1_macro": round(d_f1, 4),
    "parse_fails":      parse_fails,
    "n_samples":        len(gts),
}
with open(f"{out_dir}/{ckpt_name}_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Guardado en {out_dir}/")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

checkpoint-2750:   0%|          | 0/1000 [00:00<?, ?it/s]


Checkpoint : checkpoint-2750

── BI-RADS ──────────────────────────────────────
Accuracy : 48.9%  (489/1000)
F1-macro : 0.3973
Parse fails: 0
              precision    recall  f1-score   support

           1       0.62      0.65      0.63       494
           2       0.44      0.39      0.41       319
           3       0.13      0.16      0.15        91
           4       0.26      0.22      0.24        73
           5       0.48      0.65      0.56        23

    accuracy                           0.49      1000
   macro avg       0.39      0.41      0.40      1000
weighted avg       0.49      0.49      0.49      1000

── Density ──────────────────────────────────────
Accuracy : 80.9%  (809/1000)
F1-macro : 0.4867
              precision    recall  f1-score   support

           A       0.00      0.00      0.00         4
           B       0.57      0.73      0.64        96
           C       0.85      0.91      0.88       764
           D       0.71      0.30      0.42       136


## 16. Evaluacion de checkpoints

Evaluar cada checkpoint guardado durante training para seleccionar el mejor segun BIRADS F1 macro.
Se usa `validate_custom` con el val set completo (n_per_class=1000 = todos los disponibles).

**Por que seleccionar por F1 macro y no accuracy:** F1 macro pondera igualmente todas las clases, penalizando modelos que ignoran BIRADS 3/4/5 minoritarios.

In [ ]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# Evaluar todos los checkpoints disponibles
ckpt_dir = CHECKPOINT_DIR / 'run'
checkpoints = sorted(ckpt_dir.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[1]))
print(f'Checkpoints encontrados: {len(checkpoints)}')
for ck in checkpoints:
    print(f'  {ck.name}')

all_results = []

for ckpt in checkpoints:
    print(f'\n=== Evaluando {ckpt.name} ===')

    # Recargar modelo base + checkpoint
    model_eval = AutoModelForImageTextToText.from_pretrained(
        CONFIG['model_id'],
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        ),
        device_map='auto', torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, attn_implementation='eager',
    )
    model_eval = PeftModel.from_pretrained(model_eval, str(ckpt))
    model_eval.eval()

    results = validate_custom(
        model_eval, val_df, processor, device,
        n_per_class=1000,  # todos los disponibles en val
        max_new_tokens=124, show_examples=0
    )
    results['checkpoint'] = ckpt.name
    all_results.append(results)

    del model_eval; torch.cuda.empty_cache(); gc.collect()

# Resumen final
print('\n=== RESUMEN DE CHECKPOINTS ===')
print(f'{"Checkpoint":<25} {"BIRADS acc":>12} {"BIRADS F1":>10} {"Density acc":>12} {"Density F1":>10}')
best_ckpt = max(all_results, key=lambda x: x['birads_f1'])
for r in all_results:
    marker = ' <-- MEJOR' if r['checkpoint'] == best_ckpt['checkpoint'] else ''
    print(f'{r["checkpoint"]:<25} {r["birads_accuracy"]:>12.4f} {r["birads_f1"]:>10.4f} '
          f'{r["density_accuracy"]:>12.4f} {r["density_f1"]:>10.4f}{marker}')

print(f'\nMejor checkpoint por BIRADS F1: {best_ckpt["checkpoint"]}')